In [ ]:
# @title 1. 環境設定與 GPU 檢查
import tensorflow as tf
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import zipfile

# 設定中文字型 (Windows 微軟正黑體)
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

# 檢查 GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ 成功偵測 GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ 未偵測到 GPU，將使用 CPU (訓練會很慢)")

# 全域參數設定
IMG_SIZE = (128, 128)
BATCH_SIZE = 256
ZIP_FILENAME = 'celeba-dataset.zip' # 請確認你的檔名
EXTRACT_ROOT = os.path.join(os.getcwd(), 'celeba_data_extracted') # 解壓目錄
MODEL_SAVE_PATH = 'celeba_resnet18.h5'

print("環境設定完成。")

In [ ]:
# @title 2. 解壓縮資料集 (嚴格檢查完整性)
import os
import shutil
import zipfile

# 設定基礎路徑
ZIP_FILENAME = 'celeba-dataset.zip'
EXTRACT_ROOT = os.path.join(os.getcwd(), 'celeba_data_extracted')

# ==========================================
# 🕵️‍♀️ 搜尋輔助函數
# ==========================================
def find_image_dir(search_path):
    """搜尋包含大量圖片的資料夾"""
    for root, dirs, files in os.walk(search_path):
        jpg_count = 0
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                jpg_count += 1
                if jpg_count > 100: # 只要超過 100 張就視為圖片目錄
                    return root
    return None

def find_label_file(search_path, filename_candidates):
    """搜尋標籤檔案 (支援多種可能的檔名)"""
    for root, dirs, files in os.walk(search_path):
        for filename in filename_candidates:
            if filename in files:
                return os.path.join(root, filename)
    return None

# ==========================================
# 🚀 主執行邏輯
# ==========================================

print(f"🧐 正在檢查資料完整性 (路徑: {EXTRACT_ROOT})...")

# 1. 嘗試搜尋現有檔案
found_img_dir = find_image_dir(EXTRACT_ROOT)

# 支援 CSV 也支援 TXT (原始 CelebA 是 txt)
attr_candidates = ['list_attr_celeba.csv', 'list_attr_celeba.txt']
part_candidates = ['list_eval_partition.csv', 'list_eval_partition.txt']

found_attr = find_label_file(EXTRACT_ROOT, attr_candidates)
found_part = find_label_file(EXTRACT_ROOT, part_candidates)

# 2. 判斷是否完整
need_unzip = True

if found_img_dir and found_attr and found_part:
    # 檢查圖片數量
    total_imgs = len([f for f in os.listdir(found_img_dir) if f.lower().endswith('.jpg')])
    if total_imgs > 100000:
        print(f"✅ 現有資料完整！")
        print(f"   - 圖片: {total_imgs} 張")
        print(f"   - 屬性檔: {os.path.basename(found_attr)}")
        print(f"   - 分區檔: {os.path.basename(found_part)}")
        need_unzip = False
        IMG_DIR = found_img_dir
        CSV_ATTR = found_attr
        CSV_PARTITION = found_part
    else:
        print(f"⚠️ 圖片數量不足 ({total_imgs} 張)，準備重新解壓縮...")
else:
    print("⚠️ 資料缺失 (可能缺圖片或 CSV)，準備重新解壓縮...")
    if not found_img_dir: print("   - ❌ 找不到圖片目錄")
    if not found_attr: print("   - ❌ 找不到屬性標籤檔 (list_attr_celeba)")
    if not found_part: print("   - ❌ 找不到分區檔 (list_eval_partition)")

# 3. 執行解壓縮 (如果資料不完整)
if need_unzip:
    print(f"\n📂 開始解壓縮 '{ZIP_FILENAME}' ...")
    
    if not os.path.exists(ZIP_FILENAME):
        # 列出當前目錄檔案，幫助除錯
        print(f"當前目錄檔案: {os.listdir(os.getcwd())}")
        raise FileNotFoundError(f"❌ 找不到壓縮檔 '{ZIP_FILENAME}'，請確認檔案在當前目錄下。")

    # 清空舊目錄
    if os.path.exists(EXTRACT_ROOT):
        try:
            shutil.rmtree(EXTRACT_ROOT)
        except:
            print("⚠️ 無法刪除舊目錄，嘗試直接覆蓋...")
            
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    
    with zipfile.ZipFile(ZIP_FILENAME, 'r') as z:
        print("   正在解壓主檔案...")
        z.extractall(EXTRACT_ROOT)
        
        # 處理內層壓縮包
        for f in z.namelist():
            if 'img_align_celeba.zip' in f:
                print(f"📦 解壓內部圖片包: {f} ...")
                inner_zip = os.path.join(EXTRACT_ROOT, f)
                with zipfile.ZipFile(inner_zip, 'r') as z_in:
                    z_in.extractall(EXTRACT_ROOT)

    # 解壓後再次搜尋
    IMG_DIR = find_image_dir(EXTRACT_ROOT)
    CSV_ATTR = find_label_file(EXTRACT_ROOT, attr_candidates)
    CSV_PARTITION = find_label_file(EXTRACT_ROOT, part_candidates)

# 4. 最終確認
if not IMG_DIR or not CSV_ATTR or not CSV_PARTITION:
    # 列出所有解壓出來的檔案幫助除錯
    print("\n❌ 錯誤：解壓縮後依然找不到關鍵檔案！")
    print(f"解壓目錄 '{EXTRACT_ROOT}' 下的內容:")
    for root, dirs, files in os.walk(EXTRACT_ROOT):
        level = root.replace(EXTRACT_ROOT, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files[:5]: # 每個目錄只列出前 5 個檔案
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... (共 {len(files)} 個檔案)")
            
    raise FileNotFoundError("無法定位圖片或 CSV 檔案，請檢查 ZIP 檔案內容是否正確。")

print("\n🎉 資料準備就緒！")
print(f"✅ 圖片路徑: {IMG_DIR}")
print(f"✅ 屬性檔: {CSV_ATTR}")

In [ ]:
# @title 3. 建立資料管線 (30%驗證集 + 2x輕量增強 + 色彩修復版)
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import os
import numpy as np

# ==========================================
# 🎛️ 設定參數
# ==========================================
VAL_SPLIT_RATIO = 0.2  # 20% 驗證集
IMG_SIZE = (128, 128)  # 圖片大小

if 'BATCH_SIZE' not in globals():
    BATCH_SIZE = 256

print(f"🚀 正在建立資料管線...")
print(f"   - 驗證集比例: {VAL_SPLIT_RATIO:.0%}")
print(f"   - 資料增強: 開啟 (每張圖產生 4 種變體)")
print(f"   - Batch Size: {BATCH_SIZE}")

# ==========================================
# 1. 讀取與處理 CSV
# ==========================================
if 'CSV_ATTR' not in globals() or 'CSV_PARTITION' not in globals():
    # 嘗試自動尋找，防止變數遺失
    base_dir = os.path.join(os.getcwd(), 'celeba_data_extracted')
    for root, dirs, files in os.walk(base_dir):
        if 'list_attr_celeba.csv' in files: CSV_ATTR = os.path.join(root, 'list_attr_celeba.csv')
        if 'list_eval_partition.csv' in files: CSV_PARTITION = os.path.join(root, 'list_eval_partition.csv')
        if 'img_align_celeba' in dirs: IMG_DIR = os.path.join(root, 'img_align_celeba')

if 'CSV_ATTR' not in globals():
    raise ValueError("❌ 找不到 CSV 檔案，請重新執行【第 2 部分】！")

df_attr = pd.read_csv(CSV_ATTR)
df_part = pd.read_csv(CSV_PARTITION)

id_col_attr = df_attr.columns[0]
id_col_part = df_part.columns[0]
df_attr = df_attr.set_index(id_col_attr)
df_part = df_part.set_index(id_col_part)
df_merged = df_attr.join(df_part)

# 處理標籤 (-1 轉 0)
if 'partition' in df_merged.columns:
    attr_columns = [c for c in df_merged.columns if c != 'partition']
else:
    attr_columns = df_attr.columns.tolist()

df_merged[attr_columns] = df_merged[attr_columns].replace(-1, 0)
global LABELS_EN
LABELS_EN = attr_columns

# ==========================================
# 2. 重新劃分資料集 (Shuffle & Split)
# ==========================================
# 只取 Partition 0(Train) 和 1(Val) 來重新洗牌
df_pool = df_merged[df_merged['partition'].isin([0, 1])]
df_shuffled = df_pool.sample(frac=1, random_state=42) # 固定亂數種子

val_size = int(len(df_shuffled) * VAL_SPLIT_RATIO)
df_val = df_shuffled.iloc[:val_size]
df_train = df_shuffled.iloc[val_size:]

print(f"📊 資料集劃分完成：")
print(f"   - 訓練集: {len(df_train)} 張")
print(f"   - 驗證集: {len(df_val)} 張")

# ==========================================
# 3. 定義輕量資料增強 
# ==========================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),     # 水平翻轉
    layers.RandomRotation(0.1),          # 旋轉 ±10%
    layers.RandomZoom(0.1),              # 縮放 ±10%
    layers.RandomContrast(0.1),          # 對比度 ±10%
    layers.RandomBrightness(0.1),        # 亮度 ±10%
    # 飽和度變化 (對不同膚色泛化很有幫助)
    layers.Lambda(lambda x: tf.image.random_saturation(x, 0.8, 1.2)),
    # 色相微調
    layers.Lambda(lambda x: tf.image.random_hue(x, 0.02)),
])

# ==========================================
# 4. 定義資料生成函數
# ==========================================
def create_dataset(dataframe, img_folder, batch_size, is_training=True):
    filenames = dataframe.index.tolist()
    file_paths = [os.path.join(img_folder, fname) for fname in filenames]
    labels = dataframe[attr_columns].values.astype('float32')
    
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    
    def load_image(path, label):
        content = tf.io.read_file(path)
        img = tf.image.decode_jpeg(content, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = img / 255.0 # 正規化 0-1
        return img, label

    # 平行讀取圖片
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_training:
        # === 關鍵：重複 2 次 (2x 資料量) ===
        dataset = dataset.repeat(2)
        
        # 套用增強
        dataset = dataset.map(lambda x, y: (data_augmentation(x, training=True), y), 
                              num_parallel_calls=tf.data.AUTOTUNE)
        
        # 打亂資料
        dataset = dataset.shuffle(buffer_size=4000)
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# ==========================================
# 5. 建立 Dataset 物件
# ==========================================
train_ds = create_dataset(df_train, IMG_DIR, BATCH_SIZE, is_training=True)
val_ds = create_dataset(df_val, IMG_DIR, BATCH_SIZE, is_training=False)

print(f"✅ 資料管線 (train_ds, val_ds) 建立完成！")

In [ ]:
# @title 4. 定義模型 (ResNet18)
from tensorflow.keras import layers, models, Input

def build_resnet18_classic(input_shape, num_classes, base_filters=64):
    def resnet_block(inputs, filters, stride=1):
        # 卷積層 1
        x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        # 卷積層 2
        x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        # Shortcut 路徑
        shortcut = inputs
        if stride != 1 or inputs.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(inputs)
            shortcut = layers.BatchNormalization()(shortcut)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        return x
    inputs = Input(shape=input_shape)
    # === 初始層 ===
    x = layers.Conv2D(base_filters, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)
    # === ResNet Blocks (標準 64->128->256->512 配置) ===
    f = base_filters
    x = resnet_block(x, f)
    x = resnet_block(x, f)#
    x = resnet_block(x, f * 2, stride=2)
    x = resnet_block(x, f * 2)#
    x = resnet_block(x, f * 4, stride=2)
    x = resnet_block(x, f * 4)#
    x = resnet_block(x, f * 8, stride=2)
    x = resnet_block(x, f * 8)#
    # === 輸出層 ===
    x = layers.GlobalAveragePooling2D()(x)
    # 保留輕微 Dropout (0.25) 防止過擬合，但不影響學習
    x = layers.Dropout(0.25)(x) 
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs, name="ResNet18_Classic")
# 防呆：自動補標籤
if 'LABELS_EN' not in globals():
    LABELS_EN = list(range(40)) # 假裝有 40 個
num_classes = len(LABELS_EN)
model = build_resnet18_classic((*IMG_SIZE, 3), num_classes)
# 使用標準 binary_crossentropy，不要用加權的
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['binary_accuracy'])
print(f"✅ 模型建立完成 。")

✅ 經典 ResNet18 模型建立完成 (最穩定的版本)。


In [ ]:
# @title 5. 執行訓練 (標準模式)
from tensorflow.keras import callbacks
import tensorflow.keras.backend as K
import gc
import tkinter as tk
from tkinter import messagebox

# 0. 清理記憶體
K.clear_session()
gc.collect()

# 1. 詢問 Early Stopping
def ask_early_stopping():
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        result = messagebox.askyesno("訓練設定", "是否開啟 Early Stopping?\n\nYes: 沒進步就停 (推薦)\nNo: 跑完所有")
        root.destroy()
        return result
    except:
        return True

use_early_stop = ask_early_stopping()

# 2. 設定 Callbacks
callbacks_list = [
    callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_loss', verbose=1)
]

if use_early_stop:
    print("✅ 已開啟 Early Stopping (Patience=5)。")
    # 回歸標準的耐心值 5，讓它有機會跳出局部最佳解
    callbacks_list.append(
        callbacks.EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True, verbose=1)
    )

# 3. 執行訓練
EPOCHS = 30 

print(f"=== 開始訓練 (Epochs: {EPOCHS}) ===")
print("⚡ 使用 verbose=2 模式防止卡頓")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS, 
    callbacks=callbacks_list,
    verbose=2 
)

print(f"✅ 訓練完成！模型已儲存至 {MODEL_SAVE_PATH}")

# 繪圖
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['binary_accuracy'], label='Train Acc')
plt.plot(history.history['val_binary_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.show()

In [ ]:
# @title 6. 互動式預測 (強制刷新版：解決結果固化問題)
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import tkinter as tk
from tkinter import filedialog
from IPython.display import clear_output

# --- 1. 初始化設定 ---
clear_output(wait=True)

# 防呆預設值
if 'IMG_SIZE' not in globals(): IMG_SIZE = (128, 128)
if 'MODEL_SAVE_PATH' not in globals(): MODEL_SAVE_PATH = 'celeba_resnet18.h5'
if 'LABELS_EN' not in globals(): 
    LABELS_EN = ['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

# 中文標籤對照
LABEL_MAP_CN = {
    '5_o_Clock_Shadow': '有鬍渣', 'Arched_Eyebrows': '彎眉', 'Attractive': '有吸引力',
    'Bags_Under_Eyes': '眼袋', 'Bald': '禿頭', 'Bangs': '有瀏海', 'Big_Lips': '厚嘴唇',
    'Big_Nose': '大鼻子', 'Black_Hair': '黑髮', 'Blond_Hair': '金髮', 'Blurry': '模糊',
    'Brown_Hair': '棕髮', 'Bushy_Eyebrows': '濃眉', 'Chubby': '圓胖', 'Double_Chin': '雙下巴',
    'Eyeglasses': '戴眼鏡', 'Goatee': '山羊鬍', 'Gray_Hair': '灰/白髮', 'Heavy_Makeup': '濃妝',
    'High_Cheekbones': '高顴骨', 'Male': '男性', 'Mouth_Slightly_Open': '嘴巴微張',
    'Mustache': '八字鬍', 'Narrow_Eyes': '細長眼', 'No_Beard': '無鬍鬚', 'Oval_Face': '鵝蛋臉',
    'Pale_Skin': '皮膚白皙', 'Pointy_Nose': '尖鼻子', 'Receding_Hairline': '髮際線後退',
    'Rosy_Cheeks': '紅潤臉頰', 'Sideburns': '有鬢角', 'Smiling': '微笑', 'Straight_Hair': '直髮',
    'Wavy_Hair': '捲髮', 'Wearing_Earrings': '戴耳環', 'Wearing_Hat': '戴帽子',
    'Wearing_Lipstick': '擦口紅', 'Wearing_Necklace': '戴項鍊', 'Wearing_Necktie': '繫領帶',
    'Young': '年輕'
}

# --- 2. 載入模型 ---
model = None
try:
    if os.path.exists(MODEL_SAVE_PATH):
        # compile=False 可以加快載入速度，且避免訓練參數的干擾
        model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)
        print("✅ 模型載入成功！")
    else:
        print(f"❌ 找不到模型檔案 '{MODEL_SAVE_PATH}'。")
except Exception as e:
    print(f"❌ 模型載入失敗: {e}")

# --- 3. 預測與顯示函數 (修復快取問題) ---
def predict_and_show(image_path):
    if model is None: return

    try:
        print(f"🔍 正在讀取與分析: {os.path.basename(image_path)} ...")
        
        # A. 讀取圖片
        img_raw = tf.io.read_file(image_path)
        img_tensor = tf.image.decode_image(img_raw, channels=3)
        
        # 顯示用的圖片 (保持原始 uint8 格式)
        img_display = img_tensor.numpy().astype("uint8")
        
        # B. 預處理 (轉為 float32 並正規化)
        img_tensor = tf.cast(img_tensor, tf.float32) / 255.0
        img_tensor = tf.image.resize(img_tensor, IMG_SIZE)
        
        # ▼▼▼【關鍵修正】▼▼▼
        # 將 Tensor 轉為純 Numpy Array，斷開 TensorFlow 圖運算的快取連結
        img_input = img_tensor.numpy()
        img_input = np.expand_dims(img_input, axis=0) # 增加 batch 維度 (1, 128, 128, 3)
        # ▲▲▲
        
        # C. 預測
        probs = model.predict(img_input, verbose=0)[0]

        # D. 整理結果
        results = []
        for i, prob in enumerate(probs):
            if i < len(LABELS_EN):
                en_label = LABELS_EN[i]
                cn_label = LABEL_MAP_CN.get(en_label, en_label)
                results.append((cn_label, prob))

        # E. 顯示
        # 強制關閉舊圖表，防止重疊
        plt.close('all') 
        
        plt.figure(figsize=(10, 6))
        
        # 左邊顯示圖片
        plt.subplot(1, 2, 1)
        plt.imshow(img_display)
        plt.axis('off')
        plt.title(f"檔案: {os.path.basename(image_path)}")

        # 右邊顯示文字 (取前 10 名)
        plt.subplot(1, 2, 2)
        plt.axis('off')
        results.sort(key=lambda x: x[1], reverse=True)
        
        display_text = "預測機率 Top 10:\n\n"
        for name, p in results[:10]:
            mark = "●" if p > 0.5 else "○" 
            display_text += f"{mark} {name}: {p:.1%}\n"
            
        plt.text(0, 0.9, display_text, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"❌ 圖片分析失敗: {e}")

# --- 4. 彈出檔案選擇視窗 ---
def select_file():
    if model is None: return

    # 使用 Tkinter 彈出檔案視窗
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("👇 請在彈出的視窗中選擇圖片...")
    file_path = filedialog.askopenfilename(
        title="請選擇要測試的圖片",
        filetypes=[("Image files", "*.jpg;*.jpeg;*.png;*.bmp;*.webp")]
    )
    
    root.destroy() 

    if file_path:
        predict_and_show(file_path)
    else:
        print("⚠️ 未選擇任何檔案。")

# 執行選擇
if model is not None:
    select_file()

In [ ]:
# @title 7. 成果展現 (40x40 完整標籤關聯混淆矩陣)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
import pandas as pd

# === 1. 初始化設定 ===
if 'MODEL_SAVE_PATH' not in globals():
    MODEL_SAVE_PATH = 'celeba_resnet18.h5'

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

# 中文標籤對照
if 'LABEL_MAP_CN' not in globals():
    LABEL_MAP_CN = {
        '5_o_Clock_Shadow': '有鬍渣', 'Arched_Eyebrows': '彎眉', 'Attractive': '有吸引力',
        'Bags_Under_Eyes': '眼袋', 'Bald': '禿頭', 'Bangs': '有瀏海', 'Big_Lips': '厚嘴唇',
        'Big_Nose': '大鼻子', 'Black_Hair': '黑髮', 'Blond_Hair': '金髮', 'Blurry': '模糊',
        'Brown_Hair': '棕髮', 'Bushy_Eyebrows': '濃眉', 'Chubby': '圓胖', 'Double_Chin': '雙下巴',
        'Eyeglasses': '戴眼鏡', 'Goatee': '山羊鬍', 'Gray_Hair': '灰/白髮', 'Heavy_Makeup': '濃妝',
        'High_Cheekbones': '高顴骨', 'Male': '男性', 'Mouth_Slightly_Open': '嘴巴微張',
        'Mustache': '八字鬍', 'Narrow_Eyes': '細長眼', 'No_Beard': '無鬍鬚', 'Oval_Face': '鵝蛋臉',
        'Pale_Skin': '皮膚白皙', 'Pointy_Nose': '尖鼻子', 'Receding_Hairline': '髮際線後退',
        'Rosy_Cheeks': '紅潤臉頰', 'Sideburns': '有鬢角', 'Smiling': '微笑', 'Straight_Hair': '直髮',
        'Wavy_Hair': '捲髮', 'Wearing_Earrings': '戴耳環', 'Wearing_Hat': '戴帽子',
        'Wearing_Lipstick': '擦口紅', 'Wearing_Necklace': '戴項鍊', 'Wearing_Necktie': '繫領帶',
        'Young': '年輕'
    }

if 'LABELS_EN' not in globals():
    LABELS_EN = list(LABEL_MAP_CN.keys())

# === 2. 準備數據 ===
if 'val_ds' not in globals():
    print("❌ 錯誤：找不到驗證集 'val_ds'。請先執行【第 3 部分】。")
else:
    print("⏳ 正在計算 40x40 完整矩陣 (這需要預測所有驗證集資料)...")
    
    try:
        # 載入模型
        if 'model' not in globals():
            model = tf.keras.models.load_model(MODEL_SAVE_PATH)
        
        # 預測
        y_pred_prob = model.predict(val_ds, verbose=1)
        
        # 獲取真實標籤
        y_true = np.concatenate([y for x, y in val_ds], axis=0)
        
        # 確保長度一致
        min_len = min(len(y_true), len(y_pred_prob))
        y_true = y_true[:min_len]
        y_pred_prob = y_pred_prob[:min_len]
        
        # 轉為 0/1
        y_pred = (y_pred_prob > 0.5).astype(int)

        # === 3. 計算 40x40 共現矩陣 ===
        # 矩陣乘法: (標籤數 x 樣本數) @ (樣本數 x 標籤數) = (標籤數 x 標籤數)
        # 意義: Matrix[i, j] = 真實標籤是 i 且 預測標籤是 j 的次數
        co_matrix = np.dot(y_true.T, y_pred)
        
        # 正規化 (Normalize by Row)
        # 讓每一列的總和變成 1 (或是該類別的真實總數)，這樣變成「機率」
        # row_sums: 每個真實標籤的出現總次數
        row_sums = y_true.sum(axis=0)
        
        # 避免除以 0
        row_sums[row_sums == 0] = 1 
        
        # 進行廣播除法
        norm_matrix = co_matrix / row_sums[:, np.newaxis]

        # === 4. 繪圖 ===
        print("📊 繪製 40x40 標籤關聯混淆矩陣...")
        
        # 準備中文標籤列表
        cn_labels = [LABEL_MAP_CN.get(l, l) for l in LABELS_EN]
        
        plt.figure(figsize=(24, 20)) # 超大畫布
        
        # 繪製熱力圖
        # annot=False: 因為格子太小，顯示數字會糊成一團，建議關閉
        # 如果一定要看數字，可以改 annot=True 並把字縮很小
        sns.heatmap(norm_matrix, 
                    xticklabels=cn_labels, 
                    yticklabels=cn_labels, 
                    cmap='viridis',      # 顏色主題：亮黃色代表高相關，深紫色代表低相關
                    annot=False,         # 關閉數值顯示以保持畫面乾淨
                    linewidths=0.1,      # 格子間距
                    linecolor='gray',
                    square=True,         # 讓格子變正方形
                    cbar_kws={'label': '共現機率 (Co-occurrence Probability)'})

        plt.title('40x40 標籤關聯混淆矩陣\n(Y軸: 真實標籤 / X軸: 預測標籤)', fontsize=24, pad=20)
        plt.xlabel('AI 預測出的標籤 (Predicted)', fontsize=18)
        plt.ylabel('圖片真實的標籤 (True)', fontsize=18)
        
        plt.xticks(rotation=90, fontsize=12)
        plt.yticks(rotation=0, fontsize=12)
        
        plt.tight_layout()
        plt.show()

        print("\n🔍 圖表解讀指南：")
        print("1. 【對角線 (左上-右下)】：越亮越好。代表 AI 對該特徵的預測準確率 (Recall)。")
        print("2. 【亮黃色斑點】：代表高度相關。例如若 '山羊鬍' 列對應 '男性' 行是亮的，代表 AI 認為有鬍子的大多是男性(這是合理的)。")
        print("3. 【奇怪的亮點】：如果在不該相關的地方出現亮點(例如 '男性' 對應 '擦口紅')，代表模型學到了錯誤的偏見。")

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")

In [ ]:
# @title 8. 模型結構與參數量分析 (修正版)
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K  # 補上這行
import numpy as np                    # 補上這行
import os

# === 1. 確保模型存在 ===
# 先檢查變數是否存在，不存在則定義預設值
if 'MODEL_SAVE_PATH' not in globals():
    MODEL_SAVE_PATH = 'celeba_resnet18.h5'

if 'model' not in globals():
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"📥 正在載入模型: {MODEL_SAVE_PATH} ...")
        model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    else:
        print("❌ 找不到模型，請先建立或載入模型 (執行第 4 或 5 部分)。")
        model = None

if model is not None:
    print(f"🧐 正在分析模型：{model.name}")
    
    # === 2. 收集每一層的資訊 ===
    layers_info = []
    
    # 計算總和
    total_params = model.count_params()
    # 使用 K.count_params 計算權重
    trainable_params = np.sum([K.count_params(w) for w in model.trainable_weights])
    non_trainable_params = total_params - trainable_params
    
    # 過濾出主要的層
    for i, layer in enumerate(model.layers):
        if isinstance(layer, tf.keras.layers.InputLayer):
            continue
            
        # 獲取輸出形狀
        output_shape = layer.output_shape[1:] if layer.output_shape else "N/A"
        
        # 計算該層的「神經元/特徵點總數」
        if isinstance(output_shape, tuple) and len(output_shape) == 3:
            # Conv2D: H * W * Channels
            total_neurons = output_shape[0] * output_shape[1] * output_shape[2]
            dims = f"{output_shape[0]}x{output_shape[1]}"
            filters = output_shape[2]
        elif isinstance(output_shape, tuple) and len(output_shape) == 1:
             # Dense: Neurons
             total_neurons = output_shape[0]
             dims = "Vector"
             filters = output_shape[0]
        else:
            total_neurons = 0
            dims = "-"
            filters = "-"

        layers_info.append({
            "索引": i,
            "層類型 (Type)": layer.__class__.__name__,
            "輸出特徵圖 (H x W)": dims,
            "濾鏡/通道數 (Ch)": filters,
            "總特徵點數 (Neurons)": total_neurons,
            "參數量 (Params)": layer.count_params()
        })

    # === 3. 轉為 DataFrame 表格顯示 ===
    df_layers = pd.DataFrame(layers_info)
    
    # 顯示摘要資訊
    print("\n" + "="*50)
    print(f"📊 模型總結報告 ({model.name})")
    print("="*50)
    print(f"● 總層數 (Layers): {len(model.layers)}")
    print(f"● 總參數量 (Total Params): {total_params:,}")
    print(f"● 可訓練參數 (Trainable):  {int(trainable_params):,}")
    print(f"● 不可訓練 (Non-trainable): {int(non_trainable_params):,} (BatchNorm 統計量)")
    print("="*50 + "\n")

    # 顯示詳細表格 (只顯示有參數的層)
    df_clean = df_layers[df_layers["參數量 (Params)"] > 0].reset_index(drop=True)
    
    print("👇 重點層級列表 (只列出有權重的層)：")
    pd.set_option('display.max_rows', None) # 確保顯示所有行
    display(df_clean)

else:
    print("無法分析模型，因為模型變數為空。")

In [ ]:
# @title 獨立推論程式 (只需 .h5 檔案即可運行)
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import tkinter as tk
from tkinter import filedialog

# ==========================================
# 1. 基礎設定 (必須與訓練時一致)
# ==========================================
MODEL_PATH = 'celeba_resnet18.h5'  # 你的模型檔案路徑
IMG_SIZE = (128, 128)             # 圖片大小

# 必須手動把這 40 個標籤列出來 (因為沒有 CSV 可以讀了)
LABELS_EN = [
    '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald',
    'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair',
    'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair',
    'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache',
    'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline',
    'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings',
    'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'
]

# 中文對照表
LABEL_MAP_CN = {
    '5_o_Clock_Shadow': '有鬍渣', 'Arched_Eyebrows': '彎眉', 'Attractive': '有吸引力',
    'Bags_Under_Eyes': '眼袋', 'Bald': '禿頭', 'Bangs': '有瀏海', 'Big_Lips': '厚嘴唇',
    'Big_Nose': '大鼻子', 'Black_Hair': '黑髮', 'Blond_Hair': '金髮', 'Blurry': '模糊',
    'Brown_Hair': '棕髮', 'Bushy_Eyebrows': '濃眉', 'Chubby': '圓胖', 'Double_Chin': '雙下巴',
    'Eyeglasses': '戴眼鏡', 'Goatee': '山羊鬍', 'Gray_Hair': '灰/白髮', 'Heavy_Makeup': '濃妝',
    'High_Cheekbones': '高顴骨', 'Male': '男性', 'Mouth_Slightly_Open': '嘴巴微張',
    'Mustache': '八字鬍', 'Narrow_Eyes': '細長眼', 'No_Beard': '無鬍鬚', 'Oval_Face': '鵝蛋臉',
    'Pale_Skin': '皮膚白皙', 'Pointy_Nose': '尖鼻子', 'Receding_Hairline': '髮際線後退',
    'Rosy_Cheeks': '紅潤臉頰', 'Sideburns': '有鬢角', 'Smiling': '微笑', 'Straight_Hair': '直髮',
    'Wavy_Hair': '捲髮', 'Wearing_Earrings': '戴耳環', 'Wearing_Hat': '戴帽子',
    'Wearing_Lipstick': '擦口紅', 'Wearing_Necklace': '戴項鍊', 'Wearing_Necktie': '繫領帶',
    'Young': '年輕'
}

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 2. 載入模型
# ==========================================
print("⏳ 正在載入模型...")
try:
    # compile=False 代表我們只用來預測，不需要載入 Loss 函數和優化器
    # 這樣可以避免很多版本不相容或自定義 Loss 的錯誤
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print("✅ 模型載入成功！")
except Exception as e:
    print(f"❌ 無法載入模型: {e}")
    model = None

# ==========================================
# 3. 預測邏輯
# ==========================================
def predict_file():
    if model is None: return

    # 彈出視窗選檔案
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    file_path = filedialog.askopenfilename(title="請選擇圖片", filetypes=[("Images", "*.jpg;*.png;*.jpeg")])
    root.destroy()

    if not file_path: return

    # 讀取並預處理
    img_raw = tf.io.read_file(file_path)
    img_tensor = tf.image.decode_image(img_raw, channels=3)
    img_display = img_tensor.numpy().astype("uint8")
    
    img_tensor = tf.cast(img_tensor, tf.float32) / 255.0
    img_tensor = tf.image.resize(img_tensor, IMG_SIZE)
    img_tensor = tf.expand_dims(img_tensor, 0)

    # 推論
    probs = model.predict(img_tensor, verbose=0)[0]

    # 整理結果 (取前 10 名)
    results = []
    for i, prob in enumerate(probs):
        en = LABELS_EN[i]
        cn = LABEL_MAP_CN.get(en, en)
        results.append((cn, prob))
    
    results.sort(key=lambda x: x[1], reverse=True)

    # 畫圖
    plt.figure(figsize=(10, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img_display)
    plt.axis('off')
    plt.title("輸入圖片")

    plt.subplot(1, 2, 2)
    plt.axis('off')
    text = "預測 Top 10:\n\n"
    for name, p in results[:10]:
        mark = "●" if p > 0.5 else "○"
        text += f"{mark} {name}: {p:.1%}\n"
    
    plt.text(0, 0.9, text, fontsize=12, va='top')
    plt.show()

# 執行
if __name__ == "__main__":
    if model:
        print("👇 請在彈出視窗中選擇圖片...")
        predict_file()

In [ ]:
import tensorflow as tf
import pandas as pd
import os
import numpy as np

def analyze_model_by_blocks(model_path):
    if not os.path.exists(model_path):
        print(f"❌ 錯誤：找不到檔案 {model_path}")
        return

    print(f"📥 正在載入模型: {model_path} ...")
    try:
        model = tf.keras.models.load_model(model_path, compile=False)
    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        return

    # 準備存放 Block 的列表
    blocks_data = []
    
    # 暫存當前 Block 的資訊
    current_block = {
        "name": "Init/Stem", 
        "layers_count": 0, 
        "params": 0, 
        "output_shape": "Input",
        "filters": "N/A"
    }
    
    # 記錄上一次的 Filters 數，用來判斷是否進入新的 Stage
    last_filters = 0
    
    for layer in model.layers:
        layer_type = layer.__class__.__name__
        params = layer.count_params()
        
        # 取得輸出形狀
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list): out_shape = out_shape[0]
            display_shape = tuple(out_shape[1:]) if out_shape else "N/A"
        except:
            display_shape = "N/A"

        # -------------------------------------------------
        # 核心邏輯：偵測是否進入新的 Block (依據 Conv2D 的 Filters 變化)
        # -------------------------------------------------
        is_conv = (layer_type == 'Conv2D')
        is_dense = (layer_type == 'Dense')
        
        # 如果是 Dense 層，通常代表進入分類頭 (Classifier)
        if is_dense:
            # 先結算上一個 Block
            if current_block["layers_count"] > 0:
                blocks_data.append(current_block)
            
            # 建立分類頭 Block
            current_block = {
                "name": "Classifier (Head)",
                "layers_count": 1,
                "params": params,
                "output_shape": str(display_shape),
                "filters": f"Classes: {layer.units}"
            }
            # 強制結算分類頭，並跳出迴圈(通常是最後一層)
            blocks_data.append(current_block)
            current_block = {"layers_count": 0} # 重置以防萬一
            continue

        # 如果是 Conv2D，檢查 Filters 是否改變 (ResNet 的 Stage 特徵)
        current_filters = getattr(layer, 'filters', None)
        
        if is_conv and current_filters is not None:
            # 如果 Filters 數量變大 (例如 64 -> 128)，代表進入下一個 Stage
            if current_filters > last_filters:
                # 結算上一個 Block (如果是第一個 Conv 就不結算，視為 Stem)
                if last_filters > 0: 
                    blocks_data.append(current_block)
                
                # 初始化新 Block
                # 判斷名稱：如果是 64 filters 通常是 Stage 1
                if current_filters == 64: block_name = "Stage 1 (or Stem)"
                elif current_filters == 128: block_name = "Stage 2"
                elif current_filters == 256: block_name = "Stage 3"
                elif current_filters == 512: block_name = "Stage 4"
                else: block_name = f"Conv Block ({current_filters})"

                current_block = {
                    "name": block_name,
                    "layers_count": 0,
                    "params": 0,
                    "output_shape": str(display_shape),
                    "filters": str(current_filters)
                }
                last_filters = current_filters

        # -------------------------------------------------
        # 累加當前層的資訊到當前 Block
        # -------------------------------------------------
        if current_block.get("layers_count") is not None:
            current_block["layers_count"] += 1
            current_block["params"] += params
            # 更新輸出形狀為最新一層的形狀
            if "Pool" in layer_type or "Conv" in layer_type:
                current_block["output_shape"] = str(display_shape)

    # 處理剩餘未結算的 Block (例如 GlobalAveragePooling 接在 Stage 4 後面)
    if current_block.get("layers_count", 0) > 0 and current_block not in blocks_data:
        # 如果最後是 Pooling 層，通常歸類在 Feature Extractor 的結尾
        blocks_data.append(current_block)

    # 轉為 DataFrame
    df = pd.DataFrame(blocks_data)
    
    # 格式化數字
    df["參數量 (Params)"] = df["params"].apply(lambda x: f"{x:,}")
    df = df.drop(columns=["params"]) # 移除原始數字欄位
    df.columns = ["區塊名稱 (Block)", "層數 (Layers)", "輸出特徵圖 (Output)", "特徵深度 (Filters)", "參數量"]

    print("-" * 60)
    print(f"📊 模型總參數量: {model.count_params():,}")
    print(f"🏗️  ResNet 架構區塊分析完成")
    print("-" * 60)
    
    return df

# ==========================================
# 執行
# ==========================================
file_path = 'celeba_resnet18.h5' 

# 執行分析
df_blocks = analyze_model_by_blocks(file_path)

# 顯示結果
try:
    from IPython.display import display
    display(df_blocks)
except:
    print(df_blocks.to_string())